In [1]:
import warnings
warnings.filterwarnings("ignore")
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder, label_binarize
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
                             confusion_matrix, ConfusionMatrixDisplay, classification_report, roc_curve, auc)
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 100
RANDOM_STATE = 42


In [2]:
df = pd.read_csv("accident_data_new.csv")

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 670679 entries, 0 to 670678
Data columns (total 14 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   Index                    670679 non-null  str    
 1   Accident_Severity        670679 non-null  str    
 2   Accident Date            670679 non-null  str    
 3   Latitude                 670654 non-null  float64
 4   Light_Conditions         670679 non-null  str    
 5   District Area            670679 non-null  str    
 6   Longitude                670653 non-null  float64
 7   Number_of_Casualties     670679 non-null  int64  
 8   Number_of_Vehicles       670679 non-null  int64  
 9   Road_Surface_Conditions  669937 non-null  str    
 10  Road_Type                666099 non-null  str    
 11  Urban_or_Rural_Area      670664 non-null  str    
 12  Weather_Conditions       656328 non-null  str    
 13  Vehicle_Type             670679 non-null  str    
dtypes: float64(2), 

In [7]:
df.nunique().sum()

np.int64(1492501)

In [8]:
df.describe()

,Latitude,Longitude,Number_of_Casualties,Number_of_Vehicles
count,670654.000000,670653.000000,670679.000000,670679.000000
mean,52.554015,-1.431468,1.357013,1.831237
std,1.406956,1.383224,0.824759,0.715443
min,49.914430,-7.516225,1.000000,1.000000
25%,51.490672,-2.332413,1.000000,1.000000
50%,52.316381,-1.411817,1.000000,2.000000
75%,53.453395,-0.233365,1.000000,2.000000
max,60.757544,1.762010,68.000000,32.000000


In [9]:
df.isnull().sum()

Index                          0
Accident_Severity              0
Accident Date                  0
Latitude                      25
Light_Conditions               0
District Area                  0
Longitude                     26
Number_of_Casualties           0
Number_of_Vehicles             0
Road_Surface_Conditions      742
Road_Type                   4580
Urban_or_Rural_Area           15
Weather_Conditions         14351
Vehicle_Type                   0
dtype: int64

In [10]:
df = df.drop_duplicates(subset="Index").copy()
df["Accident Date"] = pd.to_datetime(df["Accident Date"], format="%d-%m-%Y")
df["Year"] = df["Accident Date"].dt.year
df["Month"] = df["Accident Date"].dt.month
df["DayOfWeek"] = df["Accident Date"].dt.dayofweek
df["Is_Weekend"] = (df["DayOfWeek"] >= 5).astype(int)
df["Latitude"] = df["Latitude"].fillna(df["Latitude"].median())
df["Longitude"] = df["Longitude"].fillna(df["Longitude"].median())
for col in ["Road_Surface_Conditions", "Road_Type", "Urban_or_Rural_Area", "Weather_Conditions"]:
    df[col] = df[col].fillna(df[col].mode()[0])
df["Urban_or_Rural_Area"] = df["Urban_or_Rural_Area"].replace("Unallocated", df["Urban_or_Rural_Area"].mode()[0])
df = df[~df["Vehicle_Type"].isin(["Data missing or out of range", "Ridden horse"])]
print(df.shape)
print(df.isnull().sum().sum())


(431018, 18)
0


In [11]:
df.isnull().sum()

Index                      0
Accident_Severity          0
Accident Date              0
Latitude                   0
Light_Conditions           0
District Area              0
Longitude                  0
Number_of_Casualties       0
Number_of_Vehicles         0
Road_Surface_Conditions    0
Road_Type                  0
Urban_or_Rural_Area        0
Weather_Conditions         0
Vehicle_Type               0
Year                       0
Month                      0
DayOfWeek                  0
Is_Weekend                 0
dtype: int64